# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-33: Rice iPPG (HRVCam) Exploration

### What this notebook does
Characterises the Rice iPPG dataset before anything is preprocessed, with heart-rate
variability as the question it was obtained for: what is on the disk, which of the three
per-frame clocks to trust, what the pulse oximeter actually recorded and how its samples
sit in time, how precisely beats can be timed from it, and whether the protocol's paced
deep breathing shows up in the reference as it should. This first part runs from the
recording logs and the oximeter files alone - under a megabyte per recording - so the
corpus-wide picture takes under a minute and no frame is opened. The raw frames are
examined separately.

### Dataset
Pai, Veeraraghavan, Sabharwal, *HRVCam: robust camera-based measurement of heart rate
variability* (J. Biomed. Opt. 26(2), 022707, 2021), distributed by Rice University as its
Imaging Photoplethysmography dataset. Fourteen subjects (`Subj_003`-`Subj_016`; eight male,
six female), each recorded for about two minutes in five activities: stationary with
normal breathing, reading a web page, watching a video, talking, and deep breathing paced
at 6 s in and 5 s out. One camera, a Blackfly BFLY-U3-23S6C (Sony IMX249 colour sensor) at
1280x1024 and 30 fps, gain 0, exposure set per recording, under battery-powered white LED
flood lights; a CMS50D+ finger pulse oximeter as the reference. Use requires citing the
paper, and the authors ask that participant 12's images never be used for illustration.
Redistribution is not assumed.

Per recording folder `Subj_NNN/Subj_NNN/Subj_NNN_<activity>/`:
- `cam_flea3_0/FrameNNNNN.pgm` - one raw 8-bit Bayer frame per file, 1280x1024. The colour
  PNG renderings the dataset also ships are not kept; only the raw frames are used.
- `CameraTimeLog0.txt`, `CameraCyclesLog0.txt`, `CameraFrame0.txt` - one line per frame: the
  PC's time of arrival, the camera's own clock, and the camera's frame counter.
- `Setup0.log`, `Config0.txt` - the camera settings; `window.log`, `Recorder_app.log` - the
  recorder's start, stop and oximeter events.
- `PulseOX/px1_full.mat`, `px1_partial.mat` and their `_log.txt` copies - the oximeter
  waveform with a PC time per sample. The `.pkl` copies beside them are never opened:
  unpickling a file executes code from it.

### Role in Phase 3
The one measurement of heart-rate variability from video so far, `check_hrv_from_video.py`
on MCD-rPPG, found 93% of beats but 56 ms of timing jitter against a resting RMSSD of 20-27 ms,
and closed HRV for the current model. Two parts of that harness need correcting before the
result is leaned on further: its peak finder holds a fixed 0.3 s between beats, which lets
the diastolic wave pass as a beat once the heart rate is low, and its offset search stops
at +/-0.4 s, short of the offset of up to about twenty frames (0.67 s) that MCD-rPPG
documents for its own reference. This dataset is the first in reach that was recorded for
HRV, with a reference worn throughout and a protocol that includes paced deep breathing - a
built-in positive control, since breathing at 5.5 per minute drives a large, slow swing in
heart rate that any beat-timing method should recover.

Its likely roles are evaluation ones: an HRV test set, a held-out heart-rate set, and a
breathing-rate probe at a known rate. Which of them it can carry depends first on the
reference, and that is what this part settles.

### What this notebook has to settle
1. **Is everything there?** Recordings, frames against their logs, and the oximeter files,
   before any of it is trusted.
2. **Which clock dates a frame?** Each frame carries the PC's arrival time, the camera's own
   clock and a frame counter. Beat timing needs the one that does not stall.
3. **What did the oximeter record, and when?** Its file layout, its sampling rate, whether
   samples were lost, and how to place every sample in time on the same clock as the frames.
4. **How precisely can a beat be timed from it?** A 60 Hz, 7-bit finger waveform sets a floor
   under any HRV claim built on it; the fiducial and the artefact rules decide how close to
   that floor the reference gets.
5. **Does the reference show what the protocol should produce?** HRV per activity, and the
   heart-rate swing with each paced deep breath.

No face or frame is rendered anywhere in this notebook.

## Imports & Setup

In [1]:
import ast
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linprog
from scipy.signal import butter, find_peaks, sosfiltfilt, welch

RICE_ROOT = Path(r'F:\Rice iPPG Dataset')
README_NAME = 'Read_me_IPPG Dataset.docx'
ACTIVITIES = ('stationary', 'reading', 'watching', 'talking', 'deep')   # protocol order
FRAME_DIR = 'cam_flea3_0'
FRAME_W, FRAME_H = 1280, 1024
PGM_BYTES = FRAME_W * FRAME_H + len(b'P5\n1280 1024\n255\n')   # one 8-bit frame and its header
CAMERA_LOGS = ('CameraTimeLog0.txt', 'CameraCyclesLog0.txt', 'CameraFrame0.txt')
RECORDER_LOGS = ('Setup0.log', 'Config0.txt', 'window.log', 'Recorder_app.log')
OXIMETER_FILES = ('PulseOX/px1_full.mat', 'PulseOX/px1_partial.mat',
                  'PulseOX/px1_full_log.txt', 'PulseOX/px1_partial_log.txt')

CYCLE_WRAP_S = 128.0         # the camera clock's seconds field wraps every 128 s
OX_FS = 60.0                 # CMS50D+ waveform rate; section 3 tests it against the stamps
OX_TOP_BIT = 128             # every waveform byte arrives with its top bit set
OX_FULL_SCALE = 127          # top of the 7-bit waveform; 0 and 127 are clipped values
LOSS_LOOKAHEAD = 30          # samples (0.5 s) that must all arrive a period late to accept a loss
LOSS_TOL_S = 0.003           # slack on that test, for jitter in the floor itself
HR_MIN_BPM, HR_MAX_BPM = 35.0, 180.0   # physiological limits on a beat interval
BEAT_SPACING = 0.55          # minimum spacing between beats, as a share of the dominant period
IBI_LOCAL_TOL = 0.25         # an interval this far from the median around it is an artefact
GAP_GUARD = 2                # samples either side of a beat that must be real, not interpolated
BREATH_PERIOD_S = 6.0 + 5.0  # paced deep breathing: 6 s in, 5 s out
HR_GRID_HZ = 4.0             # the heart-rate series is resampled to this rate for its spectrum
DEMO = ('003', 'deep')       # the recording printed in detail in section 3

print('Rice iPPG root:', RICE_ROOT, '| exists:', RICE_ROOT.exists())
print('README present:', (RICE_ROOT / README_NAME).exists())
print(f'oximeter taken as {OX_FS:g} Hz until section 3 measures it | beat limits '
      f'{HR_MIN_BPM:g}-{HR_MAX_BPM:g} bpm | paced breath {BREATH_PERIOD_S:g} s = '
      f'{60 / BREATH_PERIOD_S:.2f} per minute = {1 / BREATH_PERIOD_S:.3f} Hz')

Rice iPPG root: F:\Rice iPPG Dataset | exists: True
README present: True
oximeter taken as 60 Hz until section 3 measures it | beat limits 35-180 bpm | paced breath 11 s = 5.45 per minute = 0.091 Hz


## 1. Inventory
Counts what is on the disk before any of it is trusted. A recording needs three things:
its frames, the three per-frame logs that date them, and the oximeter files. The frame count
is checked against all three logs, and every frame's size against one 8-bit 1280x1024 frame -
a test on directory entries alone, so a truncated or half-written frame is caught without
opening it. The names found in each folder are tallied too, so a file that only some
recordings carry stands out.

In [2]:
SUBJECT_DIR = re.compile(r'^Subj_(\d{3})$')
RECORDING_DIR = re.compile(r'^Subj_(\d{3})_([a-z]+)$')


def frame_census(folder):
    """Returns (frames, frames not the size of one 8-bit frame) for a recording.

    Directory entries carry the file size, so every frame is checked without opening one.
    """
    path = folder / FRAME_DIR
    if not path.is_dir():
        return 0, 0
    sizes = [entry.stat().st_size for entry in os.scandir(path) if entry.name.endswith('.pgm')]
    return len(sizes), sum(size != PGM_BYTES for size in sizes)


def count_lines(path):
    """Non-empty lines in a text log, or -1 when the file is missing."""
    if not path.exists():
        return -1
    with open(path, encoding='utf-8', errors='replace') as f:
        return sum(1 for line in f if line.strip())


def scan_recordings(root=RICE_ROOT):
    """One row per recording folder, plus a tally of the names found inside them.

    Every subject folder holds a second folder of the same name, and the recordings sit
    inside that one.
    """
    rows, names = [], {}
    for subj in sorted(p for p in root.iterdir() if p.is_dir() and SUBJECT_DIR.match(p.name)):
        inner = subj / subj.name
        if not inner.is_dir():
            continue
        for folder in sorted(p for p in inner.iterdir() if p.is_dir() and RECORDING_DIR.match(p.name)):
            subject, activity = RECORDING_DIR.match(folder.name).groups()
            frames, odd_size = frame_census(folder)
            rows.append({'subject': subject, 'activity': activity, 'folder': folder,
                         'frames': frames, 'odd_size': odd_size,
                         'time_lines': count_lines(folder / 'CameraTimeLog0.txt'),
                         'cycle_lines': count_lines(folder / 'CameraCyclesLog0.txt'),
                         'counter_lines': count_lines(folder / 'CameraFrame0.txt'),
                         'missing': [f for f in CAMERA_LOGS + RECORDER_LOGS + OXIMETER_FILES
                                     if not (folder / f).exists()]})
            for entry in list(folder.iterdir()) + list((folder / 'PulseOX').glob('*')):
                key = entry.relative_to(folder).as_posix() + ('/' if entry.is_dir() else '')
                names[key] = names.get(key, 0) + 1
    return pd.DataFrame(rows), names


recordings, names = scan_recordings()
recordings['frames_match_logs'] = ((recordings.frames == recordings.time_lines)
                                   & (recordings.time_lines == recordings.cycle_lines)
                                   & (recordings.cycle_lines == recordings.counter_lines))
n_rec = len(recordings)
expected = {(s, a) for s in recordings.subject.unique() for a in ACTIVITIES}
absent = sorted(expected - set(zip(recordings.subject, recordings.activity)))
unknown = sorted(set(recordings.activity) - set(ACTIVITIES))
incomplete = recordings.loc[recordings.missing.map(len) > 0, ['subject', 'activity', 'missing']]

print(f'subjects {recordings.subject.nunique()} | recordings {n_rec} | per activity '
      f'{recordings.activity.value_counts().reindex(ACTIVITIES, fill_value=0).to_dict()}')
print('subject/activity pairs with no folder:', absent or 'none',
      '| unexpected activity names:', unknown or 'none')
print('recordings missing a log or oximeter file:', incomplete.to_dict('records') or 'none')
print()
print(f'raw frames: {recordings.frames.sum()} in {n_rec} recordings, '
      f'{recordings.frames.sum() * PGM_BYTES / 2**30:.1f} GiB | per recording '
      f'{recordings.frames.min()}-{recordings.frames.max()}')
print(f'frames that are not one 8-bit {FRAME_W}x{FRAME_H} frame by size: {recordings.odd_size.sum()}')
print(f'frame count equal to all three per-frame logs: '
      f'{int(recordings.frames_match_logs.sum())}/{n_rec}')
mismatch = recordings.loc[~recordings.frames_match_logs,
                          ['subject', 'activity', 'frames', 'time_lines', 'cycle_lines',
                           'counter_lines']]
if len(mismatch):
    print(mismatch.to_string(index=False))
print()
everywhere = sorted(k for k, v in names.items() if v == n_rec)
print(f'names in every recording ({len(everywhere)}): {", ".join(everywhere)}')
print('names in only some recordings:', {k: v for k, v in sorted(names.items()) if v < n_rec} or 'none')

subjects 14 | recordings 70 | per activity {'stationary': 14, 'reading': 14, 'watching': 14, 'talking': 14, 'deep': 14}
subject/activity pairs with no folder: none | unexpected activity names: none
recordings missing a log or oximeter file: none

raw frames: 254352 in 70 recordings, 310.5 GiB | per recording 3387-4019
frames that are not one 8-bit 1280x1024 frame by size: 0
frame count equal to all three per-frame logs: 70/70

names in every recording (19): CameraBufferSize0.txt, CameraCyclesLog0.txt, CameraFrame0.txt, CameraInfo0.log, CameraTimeLog0.txt, Config0.txt, FlyCap0.log, OT_event.log, PulseOX/, PulseOX/px1_full.mat, PulseOX/px1_full.pkl, PulseOX/px1_full_log.txt, PulseOX/px1_partial.mat, PulseOX/px1_partial.pkl, PulseOX/px1_partial_log.txt, Recorder_app.log, Setup0.log, cam_flea3_0/, window.log
names in only some recordings: none


## 2. Camera timing: three logs, and which one to trust
Every frame is dated three ways. `CameraTimeLog0.txt` holds the PC's time when the frame
arrived, `CameraCyclesLog0.txt` the camera's own clock when it was captured - seconds that
wrap every 128 s - and `CameraFrame0.txt` the camera's frame counter. The counter shows
whether a frame was dropped, and the camera clock is immune to whatever the PC was doing;
the PC's clock is the only one the oximeter shares, so it is still needed to bring the two
together.

It is not used frame by frame, though. A time written by a busy PC is late by a variable
amount and never early, so the relation between the two clocks is the lower envelope of the
points, not a least-squares line, which a burst of late stamps drags with it. `lower_line`
fits the line that lies under every point with the least total gap above it, a small linear
programme. It recovers the offset and the drift between the clocks, and what is left above
it is how late the PC was for each frame. A frame is then dated by the camera clock, mapped
onto the PC's through that line.

In [3]:
SETTINGS = {'fps_set': r'^Frame rate is ([\d.]+) fps', 'gain_db': r'^Gain is ([\d.]+)',
            'exposure_ms': r'^Exposure is ([\d.]+)'}


def read_camera_logs(folder):
    """Returns (PC arrival times, camera clock from the first frame, frame counter), in s."""
    host = np.loadtxt(folder / 'CameraTimeLog0.txt', ndmin=1)
    cycles = np.loadtxt(folder / 'CameraCyclesLog0.txt', ndmin=1)
    with open(folder / 'CameraFrame0.txt', encoding='utf-8', errors='replace') as f:
        counter = np.array([int(re.search(r'\d+', line).group()) for line in f if line.strip()])
    step = np.diff(cycles)
    step[step < 0] += CYCLE_WRAP_S
    return host, np.concatenate([[0.0], np.cumsum(step)]), counter


def lower_line(x, y):
    """(a, b) of the line a + b*x that lies under every point with the least total gap.

    Times written by a PC are late by a variable amount and never early, so a clock
    relation is the lower envelope of the points: a line constrained to lie under every
    point is held by the promptest ones alone. With x centred, the total gap depends only
    on the line's height at the centre, so the programme maximises that height. Epoch
    seconds are around 1.5e9, so both axes are shifted near zero before solving.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x0, y0 = x.mean(), y.min()
    xc = x - x0
    sol = linprog(c=[-1.0, 0.0], A_ub=np.column_stack([np.ones_like(xc), xc]), b_ub=y - y0,
                  bounds=[(None, None), (None, None)], method='highs')
    if not sol.success:
        raise RuntimeError(f'lower envelope fit failed: {sol.message}')
    a, b = sol.x
    return float(y0 + a - b * x0), float(b)


def camera_settings(folder):
    """Frame rate, gain and exposure as the camera reported them back in Setup0.log."""
    text = (folder / 'Setup0.log').read_text(encoding='utf-8', errors='replace')
    found = {key: re.search(pattern, text, flags=re.M) for key, pattern in SETTINGS.items()}
    return {key: float(m.group(1)) if m else np.nan for key, m in found.items()}


def camera_audit(row):
    """Drops, frame rate, grid deviation, clock drift and the PC's lateness for one recording.

    `grid_max_ms` is how far the camera clock strays from a uniform grid at its own rate: the
    error a store that keeps frames as a uniform sequence with one frame rate would carry.
    """
    host, clock, counter = read_camera_logs(row.folder)
    a, b = lower_line(clock, host - clock)
    lag_ms = (host - (a + (1.0 + b) * clock)) * 1000.0
    step_ms = np.diff(clock) * 1000.0
    counter_step = np.diff(counter)
    index = np.arange(clock.size)
    grid_ms = (clock - np.polyval(np.polyfit(index, clock, 1), index)) * 1000.0
    return {'subject': row.subject, 'activity': row.activity, 'frames': len(host),
            'dropped': int((counter_step[counter_step > 0] - 1).sum()),
            'backwards': int((counter_step <= 0).sum()),
            'fps': 1000.0 / float(np.median(step_ms)),
            'step_min_ms': float(step_ms.min()), 'step_max_ms': float(step_ms.max()),
            'grid_max_ms': float(np.abs(grid_ms).max()),
            'span_s': float(clock[-1]), 'drift_ppm': b * 1e6,
            'lag_med_ms': float(np.median(lag_ms)), 'lag_p99_ms': float(np.percentile(lag_ms, 99)),
            'lag_max_ms': float(lag_ms.max()), 'worst_lag_at_s': float(clock[np.argmax(lag_ms)]),
            'lag_over_10ms': float((lag_ms > 10).mean()),
            'cam_start': a, 'cam_end': a + (1.0 + b) * float(clock[-1]),
            **camera_settings(row.folder)}


camera = pd.DataFrame([camera_audit(r) for r in recordings.itertuples()])

print(f'{len(camera)} recordings | frames dropped by the camera counter: {camera.dropped.sum()} | '
      f'counter steps that go backwards: {camera.backwards.sum()}')
print(f'frame rate on the camera clock: {camera.fps.min():.3f}-{camera.fps.max():.3f} fps | '
      f'frame interval {camera.step_min_ms.min():.2f}-{camera.step_max_ms.max():.2f} ms | '
      f'duration {camera.span_s.min():.1f}-{camera.span_s.max():.1f} s')
print(f'camera clock against a uniform grid at its own rate: worst {camera.grid_max_ms.max():.3f} ms '
      f'in any recording')
print(f'camera clock against the PC clock: {camera.drift_ppm.median():+.0f} ppm '
      f'in the median recording')
print(f'PC arrival lag above the envelope, median recording: '
      f'median {camera.lag_med_ms.median():.2f} ms, '
      f'99th percentile {camera.lag_p99_ms.median():.2f} ms, worst {camera.lag_max_ms.median():.2f} ms')
late = camera[camera.lag_max_ms > 20].sort_values('lag_max_ms', ascending=False)
print(f'recordings whose PC stamps run more than 20 ms late at some point: {len(late)}')
if len(late):
    print(late[['subject', 'activity', 'span_s', 'worst_lag_at_s', 'lag_med_ms', 'lag_p99_ms',
                'lag_max_ms', 'lag_over_10ms']].round(3).to_string(index=False))
print()
print('settings reported back by the camera: frame rate',
      camera.fps_set.round(2).value_counts().to_dict(),
      '| gain', camera.gain_db.round(2).value_counts().to_dict())
print('exposure (ms, rounded) per subject and activity:')
print(camera.assign(exposure=camera.exposure_ms.round(0).astype('Int64'))
      .pivot(index='subject', columns='activity', values='exposure')
      .reindex(columns=list(ACTIVITIES)).to_string())

70 recordings | frames dropped by the camera counter: 0 | counter steps that go backwards: 0
frame rate on the camera clock: 30.017-30.018 fps | frame interval 33.27-33.36 ms | duration 112.8-133.9 s
camera clock against a uniform grid at its own rate: worst 0.046 ms in any recording
camera clock against the PC clock: -11 ppm in the median recording
PC arrival lag above the envelope, median recording: median 0.09 ms, 99th percentile 0.63 ms, worst 1.75 ms
recordings whose PC stamps run more than 20 ms late at some point: 4
subject activity  span_s  worst_lag_at_s  lag_med_ms  lag_p99_ms  lag_max_ms  lag_over_10ms
    015  reading 122.061         117.597       0.140    1660.538    2394.587          0.031
    005  reading 119.562          35.379      74.067     103.631     104.347          0.882
    006  reading 121.394          58.865       7.826      44.429      45.671          0.452
    005 watching 120.462          28.516       8.548      25.716      26.433          0.446

settings r

## 3. The oximeter: its files, its clock, and the samples it lost
The CMS50D+ streams a plethysmographic waveform over a serial link, and the recorder stored it
in `px1_full.mat`: `pulseOxRecord`, one byte per sample with its top bit set (0-127 once 128
is removed), `pulseOxTime`, the PC's time when each sample was read, and `numPulseSample`. No
pulse rate, SpO2 or beat marker is kept, so the heart rate has to come from the waveform.
`px1_partial.mat` and the text log are checked against it.

The stamps record when the PC read each sample, not when the device took it. The reader
collects the serial stream in bursts - intervals of 20, 20 and 10 ms where the device delivers
one sample every 16.7 ms - and it also loses samples, one or two at a time. Both matter for
HRV. Dating samples by their stamps adds the reading jitter to every beat time; dating them by
their index alone, at 60 Hz, puts every sample after a loss one period early, and the error
accumulates over the recording.

The device's own clock can be rebuilt because it is regular and the losses are whole samples.
Against a 60 Hz grid, the stamps minus their slot times form a floor of prompt reads with late
ones above it, and that floor steps up by exactly one period at each lost sample.
`sample_slots` walks the stamps and accepts a loss only when every one of the next half second
of samples arrives at least one period (less 3 ms) above the floor of the half second before
it, so a single late read never passes for a loss. A step down - a sample apparently taken
before its predecessor - is impossible under this model; it is counted rather than repaired,
and a count above zero would mean the model does not fit that recording. `lower_line` then
fits the floor against the slots, which gives every sample a time on the PC's clock and the
device's true rate; the stamp minus that time is the read delay, zero or more by construction.

The rebuilt times also say how much of each recording the reference covers. Three recorder
logs contain `Cannot read pulse-ox`: the oximeter link failed before the camera stopped.

A positive control closes the section. Known losses are injected into a clean 60 Hz grid at
1%, three times the dataset's own rate, each recording's rebuilt read delays are added back,
and the rule is asked to find them. A loss placed a sample or two early or late only misdates
samples next to the interpolated slot, whose beats section 4 drops anyway; what would matter is
a misplacement further away, or a count that comes out wrong. The delays come from the rebuild
itself, so this tests the rule against the jitter it actually faces, not against an
independent truth.

In [4]:
def read_oximeter(folder):
    """Returns (waveform bytes, PC read times in s, the .mat contents) from px1_full.mat."""
    mat = sio.loadmat(folder / 'PulseOX' / 'px1_full.mat', squeeze_me=True)
    values = np.atleast_1d(mat['pulseOxRecord']).astype(np.int64)
    stamps = np.atleast_1d(mat['pulseOxTime']).astype(np.float64)
    return values, stamps, mat


def oximeter_files(folder, values, stamps, mat):
    """Whether the full record agrees with its count, its text log and the partial copy."""
    part = sio.loadmat(folder / 'PulseOX' / 'px1_partial.mat', squeeze_me=True)
    part_values = np.atleast_1d(part['pulseOxRecord']).astype(np.int64)
    part_stamps = np.atleast_1d(part['pulseOxTime']).astype(np.float64)
    # The text log is a Python list literal; literal_eval parses literals and runs nothing.
    text = (folder / 'PulseOX' / 'px1_full_log.txt').read_text(encoding='utf-8', errors='replace')
    log = np.asarray(ast.literal_eval(text.strip()), dtype=np.float64)
    n_part = len(part_values)
    return {'variables': ' '.join(sorted(k for k in mat if not k.startswith('__'))),
            'count_matches': int(mat['numPulseSample']) == len(values),
            'log_matches': (len(log) == len(stamps)
                            and bool(np.allclose(log, stamps, rtol=0, atol=1e-6))),
            'partial_is_prefix': (n_part <= len(values)
                                  and bool(np.array_equal(part_values, values[:n_part]))
                                  and bool(np.allclose(part_stamps, stamps[:n_part],
                                                       rtol=0, atol=1e-6))),
            'partial_shorter_by': len(values) - n_part,
            'top_bit_set': bool(np.all((values >= OX_TOP_BIT) & (values < 2 * OX_TOP_BIT)))}


def sample_slots(stamps, fs=OX_FS, lookahead=LOSS_LOOKAHEAD, tol=LOSS_TOL_S):
    """Places every read sample in its slot on the device's own grid.

    Returns (slots, steps_down): slots[k] is k plus the samples lost before sample k. A
    stamp minus its slot time sits on a floor of prompt reads, with late reads above it,
    and the floor steps up one period at each lost sample. A loss is accepted only when all
    of the next `lookahead` samples sit at least a period (less `tol`) above the floor of the
    `lookahead` before them, so one late read never passes for a loss. A step down is
    impossible under this model; the number of such events is returned, not repaired.
    """
    period = 1.0 / fs
    n = stamps.size
    slots = np.empty(n, dtype=np.int64)
    floor = np.empty(n)
    below = np.zeros(n, dtype=bool)
    lost = 0
    for k in range(n):
        hi = min(n, k + lookahead)
        ahead = stamps[k:hi] - (np.arange(k, hi) + lost) * period
        before = ahead.min() if k == 0 else floor[max(0, k - lookahead):k].min()
        while hi - k >= 5 and ahead.min() - before >= period - tol:
            lost += 1
            ahead = ahead - period
        below[k] = ahead.min() - before <= -(period - tol)
        slots[k] = k + lost
        floor[k] = stamps[k] - slots[k] * period
    steps_down = int(below[0]) + int((below[1:] & ~below[:-1]).sum())
    return slots, steps_down


def rebuild_clock(stamps, fs=OX_FS):
    """Slots, sample times on the PC clock, read delays and the device's measured rate."""
    slots, steps_down = sample_slots(stamps, fs)
    grid_s = slots / fs
    a, b = lower_line(grid_s, stamps - grid_s)
    times = a + (1.0 + b) * grid_s
    return {'slots': slots, 'times': times, 'delay': stamps - times,
            'rate_hz': fs / (1.0 + b), 'steps_down': steps_down}


def oximeter_audit(row, cam):
    """File checks, clock rebuild and coverage of one recording: (audit row, reference)."""
    values, stamps, mat = read_oximeter(row.folder)
    clock = rebuild_clock(stamps)
    gaps = np.diff(clock['slots']) - 1
    index = np.arange(stamps.size)
    straight = np.polyval(np.polyfit(index, stamps, 1), index)
    window = (row.folder / 'window.log').read_text(encoding='utf-8', errors='replace')
    audit = {'subject': row.subject, 'activity': row.activity, 'samples': stamps.size,
             **oximeter_files(row.folder, values, stamps, mat),
             'span_s': float(stamps[-1] - stamps[0]),
             'stamp_rate_hz': (stamps.size - 1) / float(stamps[-1] - stamps[0]),
             'device_ppm': (clock['rate_hz'] / OX_FS - 1.0) * 1e6,
             'lost': int(gaps.sum()), 'loss_events': int((gaps > 0).sum()),
             'longest_gap': int(gaps.max()) if gaps.size else 0,
             'lost_share': float(gaps.sum() / (clock['slots'][-1] + 1)),
             'steps_down': clock['steps_down'],
             'delay_med_ms': float(np.median(clock['delay']) * 1000.0),
             'delay_p95_ms': float(np.percentile(clock['delay'], 95) * 1000.0),
             'delay_max_ms': float(clock['delay'].max() * 1000.0),
             'index_drift_s': float((stamps[-1] - stamps[0]) - (stamps.size - 1) / OX_FS),
             'stamp_wander_ms': float(np.ptp(stamps - straight) * 1000.0),
             'read_failures': window.count('Cannot read pulse-ox'),
             'starts_after_camera_s': float(clock['times'][0] - cam.cam_start),
             'ends_before_camera_s': float(cam.cam_end - clock['times'][-1])}
    return audit, (values - OX_TOP_BIT, clock['slots'], clock['times'])


REFERENCE = {}
rows = []
for r in recordings.itertuples():
    cam = camera.loc[(camera.subject == r.subject) & (camera.activity == r.activity)].iloc[0]
    audit, reference = oximeter_audit(r, cam)
    REFERENCE[(r.subject, r.activity)] = reference
    rows.append(audit)
oximeter = pd.DataFrame(rows)
n_ox = len(oximeter)

print('variables in px1_full.mat:', oximeter.variables.value_counts().to_dict())
print(f'numPulseSample equals the record length: {int(oximeter.count_matches.sum())}/{n_ox} | '
      f'the text log equals pulseOxTime: {int(oximeter.log_matches.sum())}/{n_ox} | '
      f'top bit set in every byte: {int(oximeter.top_bit_set.sum())}/{n_ox}')
print(f'px1_partial.mat is a leading part of px1_full.mat: '
      f'{int(oximeter.partial_is_prefix.sum())}/{n_ox}, '
      f'shorter in {int((oximeter.partial_shorter_by > 0).sum())} (by up to '
      f'{oximeter.partial_shorter_by.max()} samples), so the full record is the complete one')
print()
print(f'samples per recording {oximeter.samples.min()}-{oximeter.samples.max()} over '
      f'{oximeter.span_s.min():.1f}-{oximeter.span_s.max():.1f} s; samples per second of stamps '
      f'{oximeter.stamp_rate_hz.min():.2f}-{oximeter.stamp_rate_hz.max():.2f}')
print(f'device rate on the rebuilt clock: {OX_FS:g} Hz {oximeter.device_ppm.median():+.0f} ppm in the '
      f'median recording (interquartile {oximeter.device_ppm.quantile(0.25):+.0f} to '
      f'{oximeter.device_ppm.quantile(0.75):+.0f})')
print(f'lost samples: {oximeter.lost.sum()} in {int((oximeter.lost > 0).sum())}/{n_ox} recordings | '
      f'per recording median {oximeter.lost.median():.0f}, '
      f'range {oximeter.lost.min()}-{oximeter.lost.max()} | '
      f'every loss a single sample in {int((oximeter.longest_gap <= 1).sum())} recordings, '
      f'longest gap {oximeter.longest_gap.max()} samples')
print(f'steps down, impossible under the loss model: {oximeter.steps_down.sum()}')
print(f'read delay above the rebuilt clock, median recording: '
      f'median {oximeter.delay_med_ms.median():.1f} ms, '
      f'95th percentile {oximeter.delay_p95_ms.median():.1f} ms')
drift, wander = oximeter.index_drift_s, oximeter.stamp_wander_ms
print(f'dating samples by index alone ends {drift.median():.2f} s early in the median recording, '
      f'up to {drift.max():.2f} s; a straight line through the stamps leaves {wander.median():.0f} ms '
      f'of wander peak to peak (up to {wander.max():.0f} ms)')
_, demo_slots, demo_times = REFERENCE[DEMO]
demo_gaps = np.diff(demo_slots) - 1
demo_losses = [f'{demo_times[i + 1] - demo_times[0]:.1f} s'
               + (f' ({demo_gaps[i]} samples)' if demo_gaps[i] > 1 else '')
               for i in np.flatnonzero(demo_gaps)]
print(f'{DEMO[0]}/{DEMO[1]}: losses at ' + ', '.join(demo_losses))
print()
weak = oximeter[(oximeter.lost_share > 0.01) | (oximeter.longest_gap > 3)
                | (oximeter.delay_med_ms > 20) | (oximeter.read_failures > 0)]
print('recordings with more than 1% lost, a gap over 3 samples, a median read delay over 20 ms, '
      'or a failed oximeter read:')
print(weak[['subject', 'activity', 'lost', 'lost_share', 'longest_gap', 'delay_med_ms', 'read_failures',
            'ends_before_camera_s']].round(3).to_string(index=False))
print()
print(f'the reference starts {oximeter.starts_after_camera_s.min():.2f}-'
      f'{oximeter.starts_after_camera_s.max():.2f} s '
      f'after the first frame and ends {oximeter.ends_before_camera_s.min():+.2f} to '
      f'{oximeter.ends_before_camera_s.max():+.2f} s before the last one')
uncovered = oximeter[oximeter.ends_before_camera_s > 1]
print('recordings whose last second or more of frames has no reference:',
      uncovered[['subject', 'activity', 'ends_before_camera_s', 'read_failures']]
      .round(2).to_dict('records') or 'none')

variables in px1_full.mat: {'numPulseSample pulseOxRecord pulseOxTime': 70}
numPulseSample equals the record length: 70/70 | the text log equals pulseOxTime: 70/70 | top bit set in every byte: 70/70
px1_partial.mat is a leading part of px1_full.mat: 70/70, shorter in 12 (by up to 265 samples), so the full record is the complete one

samples per recording 5692-8002 over 112.0-133.7 s; samples per second of stamps 47.61-59.95
device rate on the rebuilt clock: 60 Hz -15 ppm in the median recording (interquartile -17 to +16)
lost samples: 2803 in 70/70 recordings | per recording median 20, range 6-1482 | every loss a single sample in 62 recordings, longest gap 14 samples
steps down, impossible under the loss model: 0
read delay above the rebuilt clock, median recording: median 5.3 ms, 95th percentile 9.8 ms
dating samples by index alone ends 0.32 s early in the median recording, up to 24.69 s; a straight line through the stamps leaves 122 ms of wander peak to peak (up to 7820 ms)
003/deep:

In [5]:
CONTROL_LOSS_RATE = 0.01   # injected losses per sample; the dataset's own rate is about 0.3%
CONTROL_RUNS = 3           # repetitions per recording, each with fresh loss positions


def recovery_check(delays, rng, loss_rate=CONTROL_LOSS_RATE):
    """Known losses on a clean grid plus a recording's own read delays, then the rule again.

    Returns (losses injected, misplacements further than GAP_GUARD samples from any true
    loss, error in the final count).
    """
    gaps = (rng.random(delays.size - 1) < loss_rate).astype(np.int64)
    truth = np.concatenate([[0], np.cumsum(1 + gaps)])
    stamps = np.maximum.accumulate(truth / OX_FS + delays)
    slots, _ = sample_slots(stamps)
    error = slots - truth
    changes = np.flatnonzero(np.diff(error)) + 1
    loss_at = np.flatnonzero(gaps) + 1
    if changes.size and loss_at.size:
        far = np.min(np.abs(changes[:, None] - loss_at[None, :]), axis=1) > GAP_GUARD
    else:
        far = np.ones(changes.size, dtype=bool)
    return loss_at.size, int(far.sum()), abs(int(error[-1]))


rng = np.random.default_rng(0)
rows = []
for r in recordings.itertuples():
    _, slots, times = REFERENCE[(r.subject, r.activity)]
    delays = read_oximeter(r.folder)[1] - times
    runs = np.array([recovery_check(delays, rng) for _ in range(CONTROL_RUNS)])
    rows.append({'subject': r.subject, 'activity': r.activity, 'injected': int(runs[:, 0].sum()),
                 'misplaced': int(runs[:, 1].sum()), 'count_error': int(runs[:, 2].sum())})
control = pd.DataFrame(rows).merge(oximeter[['subject', 'activity', 'delay_med_ms', 'delay_p95_ms']],
                                   on=['subject', 'activity'])
control['error_rate'] = (control.misplaced + control.count_error) / control.injected
print(f'positive control: {control.injected.sum()} losses injected at {CONTROL_LOSS_RATE:.0%}, '
      f'{CONTROL_RUNS} runs into each of {len(control)} recordings, '
      f'each run with the recording\'s own read delays')
print(f'final count wrong in {int((control.count_error > 0).sum())} recordings (summed error '
      f'{control.count_error.sum()}) | misplaced more than {GAP_GUARD} samples from any true loss: '
      f'{control.misplaced.sum()}, '
      f'{control.misplaced.sum() / (len(control) * CONTROL_RUNS):.2f} per run')
shaky = control[control.error_rate > 0.02].sort_values('error_rate', ascending=False)
print('recordings where the rule mishandled more than 2% of the injected losses, '
      'with their read delays:')
print(shaky[['subject', 'activity', 'misplaced', 'count_error', 'error_rate', 'delay_med_ms',
             'delay_p95_ms']].round(3).to_string(index=False) if len(shaky) else '  none')

positive control: 15082 losses injected at 1%, 3 runs into each of 70 recordings, each run with the recording's own read delays
final count wrong in 28 recordings (summed error 65) | misplaced more than 2 samples from any true loss: 59, 0.28 per run
recordings where the rule mishandled more than 2% of the injected losses, with their read delays:
subject   activity  misplaced  count_error  error_rate  delay_med_ms  delay_p95_ms
    006    reading         15           14       0.133        22.573        70.863
    006 stationary          6            5       0.065        18.020        25.797
    005 stationary          5            6       0.045        10.324        38.533
    005    reading          1            7       0.041        77.698       108.308
    015   watching          4            3       0.033        11.191        22.505
    005   watching          2            4       0.031        13.447        44.360
    013   watching          4            1       0.021         6.318   

## 4. Reference beats from a 60 Hz, 7-bit finger PPG
The waveform goes onto the rebuilt 60 Hz grid, with lost samples interpolated and marked.
Three choices then decide how precise the beat times are.

**The fiducial.** Two are measured: the steepest point of the upstroke, and the systolic peak.
The upstroke is the sharpest feature of a finger pulse and the least changed by the reflected
wave; the peak is blunt, and where the diastolic wave rises to nearly the same height the
maximum can move from one to the other between beats. Both are refined between samples with
the three-point parabola the HRV scripts already use.

**The minimum spacing between beats.** `beat_times` in `check_hrv_sampling_ceiling.py` holds a
fixed 0.3 s, which lets the diastolic wave pass as a second beat once the heart rate is low
enough. Here the spacing is 55% of the recording's own dominant beat period, read from its
spectrum, which still admits the shortest intervals of a deep-breathing swing.

**What counts as an artefact.** An interval is dropped when it lies outside 35-180 bpm, when
either of its beats has an interpolated sample, or a clipped one at 0 or 127, within two
samples of its upstroke, or when it differs by more than 25% from the median of the seven
intervals centred on it. Successive differences are taken only between two kept intervals that
are neighbours, so a single artefact cannot manufacture a large difference.

The precision floor comes from simulation: a pulse train with known beat times, sampled at
60 Hz with a random phase, rounded to whole levels at the beat amplitudes this dataset shows,
and timed with the same filters and parabola. Its timing error converts to RMSSD in
quadrature. A successive difference of intervals combines three beat times,
t[i] - 2t[i+1] + t[i+2], so independent timing errors with standard deviation s add a noise
RMSSD of q = sqrt(6)·s, and a true RMSSD r is measured as sqrt(r² + q²). The error is the
quadrature difference sqrt(r² + q²) - r; the sampling-ceiling script's `predicted` column
reports q itself, which overstates it.

In [6]:
BEAT_BAND = butter(2, [0.5, 8.0], btype='band', fs=OX_FS, output='sos')
HR_BAND = butter(2, [HR_MIN_BPM / 60.0, HR_MAX_BPM / 60.0], btype='band', fs=OX_FS, output='sos')


def on_grid(wave, slots):
    """The waveform on the device's 60 Hz grid, with lost samples interpolated and marked."""
    grid = np.full(int(slots[-1]) + 1, np.nan)
    grid[slots] = wave
    missing = np.isnan(grid)
    if missing.any():
        idx = np.arange(grid.size)
        grid[missing] = np.interp(idx[missing], idx[~missing], grid[~missing])
    return grid, missing


def dominant_period(grid, fs=OX_FS):
    """The recording's dominant beat period in samples, from a 16 s Welch spectrum."""
    f, p = welch(sosfiltfilt(HR_BAND, grid - grid.mean()), fs=fs, nperseg=int(16 * fs))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    return fs / f[band][np.argmax(p[band])]


def refine(y, i):
    """Sub-sample position of a maximum from a parabola through three points.

    The same estimator as `refine` in check_hrv_sampling_ceiling.py: the index comes back
    unchanged at an edge, on a flat top, or when the vertex would leave the sample's cell.
    """
    if i <= 0 or i >= len(y) - 1:
        return float(i)
    left, centre, right = float(y[i - 1]), float(y[i]), float(y[i + 1])
    denominator = left - 2.0 * centre + right
    if abs(denominator) < 1e-12:
        return float(i)
    offset = 0.5 * (left - right) / denominator
    return float(i) + (offset if abs(offset) <= 0.5 else 0.0)


def find_beats(grid, missing, fs=OX_FS):
    """One row per beat: steepest-upstroke and systolic-peak times, amplitude and flags.

    A candidate must climb at least 40% as steeply as the median of the candidates around
    it, which drops ripples on the downstroke and noise between beats without a fixed
    threshold that the breathing-driven amplitude swing would defeat.
    """
    x = sosfiltfilt(BEAT_BAND, grid - grid.mean())
    slope = np.gradient(x)
    period = dominant_period(grid, fs)
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return pd.DataFrame(columns=['slope_s', 'peak_s', 'amplitude', 'near_gap', 'clipped']), period
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    rows = []
    for i in idx:
        top = min(grid.size - 1, i + int(0.45 * period))
        j = i + int(np.argmax(x[i:top + 1]))
        around = grid[max(0, i - GAP_GUARD):i + GAP_GUARD + 1]
        trough = grid[max(0, i - int(0.4 * period)):i + 1].min()
        rows.append({'slope_s': refine(slope, int(i)) / fs, 'peak_s': refine(x, j) / fs,
                     'amplitude': float(grid[i:top + 1].max() - trough),
                     'near_gap': bool(missing[max(0, i - GAP_GUARD):i + GAP_GUARD + 1].any()),
                     'clipped': bool(((around <= 0) | (around >= OX_FULL_SCALE)).any())})
    return pd.DataFrame(rows), period


def intervals(beats):
    """Beat-to-beat intervals for both fiducials, with the artefact rules applied.

    `kept` marks the intervals HRV is computed from; `reason` says why the others were
    dropped, in the order the rules are applied.
    """
    t = beats.slope_s.to_numpy()
    ibi = np.diff(t)
    flagged = (beats.near_gap | beats.clipped).to_numpy()
    in_range = (ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)
    clean = ~flagged[:-1] & ~flagged[1:]
    candidate = in_range & clean
    local = (pd.Series(np.where(candidate, ibi, np.nan))
             .rolling(7, center=True, min_periods=3).median().to_numpy())
    consistent = np.abs(ibi - local) <= IBI_LOCAL_TOL * local
    return pd.DataFrame({'t': t[1:], 'ibi': ibi, 'ibi_peak': np.diff(beats.peak_s.to_numpy()),
                         'kept': candidate & consistent,
                         'reason': np.select([~in_range, ~clean, ~consistent],
                                             ['range', 'gap or clip', 'local'], default='')})


def hrv_metrics(table, column='ibi'):
    """Mean heart rate, SDNN, RMSSD and pNN50 from the kept intervals of one fiducial."""
    ibi = table[column].to_numpy()
    kept = table.kept.to_numpy()
    pair = kept[1:] & kept[:-1]
    diffs = np.diff(ibi)[pair]
    values = ibi[kept]
    if values.size < 8 or diffs.size < 8:
        return dict.fromkeys(('hr_bpm', 'sdnn_ms', 'rmssd_ms', 'pnn50'), np.nan)
    return {'hr_bpm': 60.0 / values.mean(), 'sdnn_ms': values.std(ddof=1) * 1000.0,
            'rmssd_ms': float(np.sqrt(np.mean(diffs ** 2)) * 1000.0),
            'pnn50': float(np.mean(np.abs(diffs) > 0.05))}


BEATS = {}
rows = []
for (subject, activity), (wave, slots, _) in REFERENCE.items():
    grid, missing = on_grid(wave.astype(np.float64), slots)
    beats, period = find_beats(grid, missing)
    table = intervals(beats)
    BEATS[(subject, activity)] = table
    rows.append({'subject': subject, 'activity': activity,
                 'hr_spectral_bpm': 60.0 * OX_FS / period, 'beats': len(beats),
                 'intervals': len(table), 'kept_share': float(table.kept.mean()),
                 'drop_range': int((table.reason == 'range').sum()),
                 'drop_gap_clip': int((table.reason == 'gap or clip').sum()),
                 'drop_local': int((table.reason == 'local').sum()),
                 'amplitude': float(beats.amplitude.median()),
                 'at_floor': float((wave == 0).mean()),
                 'at_ceiling': float((wave == OX_FULL_SCALE).mean()),
                 'rmssd_slope_ms': hrv_metrics(table)['rmssd_ms'],
                 'rmssd_peak_ms': hrv_metrics(table, 'ibi_peak')['rmssd_ms']})
beat_audit = pd.DataFrame(rows)

print(f'waveform at the floor of its range (0): median {beat_audit.at_floor.median():.2%} of samples, '
      f'worst {beat_audit.at_floor.max():.2%} | at the ceiling ({OX_FULL_SCALE}): median '
      f'{beat_audit.at_ceiling.median():.2%}, worst {beat_audit.at_ceiling.max():.2%}')
print(f'beat amplitude, recording medians: '
      f'{beat_audit.amplitude.min():.0f}-{beat_audit.amplitude.max():.0f} '
      f'levels of {OX_FULL_SCALE} (median {beat_audit.amplitude.median():.0f})')
print(f'beats per recording {beat_audit.beats.min()}-{beat_audit.beats.max()} | intervals kept: median '
      f'{beat_audit.kept_share.median():.1%}, lowest {beat_audit.kept_share.min():.1%}')
print(f'dropped, all recordings: outside {HR_MIN_BPM:g}-{HR_MAX_BPM:g} bpm '
      f'{beat_audit.drop_range.sum()}, '
      f'near a lost sample or clipped {beat_audit.drop_gap_clip.sum()}, '
      f'more than {IBI_LOCAL_TOL:.0%} from the local median {beat_audit.drop_local.sum()}')
print('\nfewest intervals kept, worst 6:')
print(beat_audit.nsmallest(6, 'kept_share')[['subject', 'activity', 'beats', 'kept_share',
                                             'drop_gap_clip', 'drop_local', 'at_floor']]
      .round(3).to_string(index=False))

waveform at the floor of its range (0): median 0.47% of samples, worst 8.15% | at the ceiling (127): median 0.00%, worst 0.57%
beat amplitude, recording medians: 35-65 levels of 127 (median 52)
beats per recording 101-226 | intervals kept: median 96.7%, lowest 50.3%
dropped, all recordings: outside 35-180 bpm 11, near a lost sample or clipped 499, more than 25% from the local median 24

fewest intervals kept, worst 6:
subject   activity  beats  kept_share  drop_gap_clip  drop_local  at_floor
    006 stationary    160       0.503             76           3     0.003
    005    talking    201       0.860             28           0     0.048
    008   watching    110       0.862             12           2     0.026
    004       deep    218       0.866             26           2     0.081
    013   watching    182       0.873             23           0     0.031
    014   watching    133       0.879             15           1     0.018


In [7]:
def beat_template(t):
    """A finger-PPG beat on a 0-1 scale: the systolic wave and a smaller diastolic one."""
    return (np.exp(-0.5 * ((t - 0.16) / 0.055) ** 2)
            + 0.45 * np.exp(-0.5 * ((t - 0.40) / 0.08) ** 2))


def timing_error_ms(amplitude, quantise, sub_sample, n_beats=400, seed=0, fs=OX_FS):
    """Standard deviation of the steepest-upstroke time on a simulated 60 Hz pulse train.

    The train is built on a continuous time axis, sampled with a random phase, optionally
    rounded to whole levels, and timed exactly as the reference is. The constant offset
    between a beat's onset and its steepest point is removed, so what remains is the
    sampling and rounding error alone.
    """
    rng = np.random.default_rng(seed)
    onsets = 1.0 + np.cumsum(0.8 + 0.05 * rng.standard_normal(n_beats))
    phase = rng.uniform(0.0, 1.0 / fs)
    t = np.arange(phase, onsets[-1] + 2.0, 1.0 / fs)
    wave = np.zeros_like(t)
    for onset in onsets:
        m = (t >= onset - 0.1) & (t < onset + 1.0)
        wave[m] += beat_template(t[m] - onset)
    wave = 10.0 + amplitude * wave
    if quantise:
        wave = np.round(wave)
    slope = np.gradient(sosfiltfilt(BEAT_BAND, wave - wave.mean()))
    idx, _ = find_peaks(slope, distance=int(BEAT_SPACING * 0.8 * fs))
    idx = idx[slope[idx] >= 0.4 * np.median(slope[idx])]
    est = np.array([refine(slope, int(i)) if sub_sample else float(i) for i in idx]) / fs + phase
    onset = onsets[np.clip(np.searchsorted(onsets, est - 0.4), 0, onsets.size - 1)]
    err = est - onset
    err = err[np.abs(err - np.median(err)) < 0.1]
    return float(np.std(err - np.median(err)) * 1000.0)


levels = sorted({int(round(beat_audit.amplitude.quantile(0.1))),
                 int(round(beat_audit.amplitude.median()))})
rmssd_points = [float(beat_audit.rmssd_slope_ms.quantile(q)) for q in (0.1, 0.5)]
rows = []
for amplitude in levels:
    for quantise in (False, True):
        for sub_sample in (False, True):
            sd = timing_error_ms(amplitude, quantise, sub_sample)
            q = np.sqrt(6.0) * sd
            rows.append({'amplitude': amplitude, 'rounded_to_levels': quantise,
                         'sub_sample': sub_sample,
                         'timing_sd_ms': sd, 'noise_rmssd_ms': q,
                         **{f'error_at_{r:.0f}ms': np.sqrt(r ** 2 + q ** 2) - r for r in rmssd_points}})
floor = pd.DataFrame(rows)

print(f'timing floor on a simulated 60 Hz pulse train (beat amplitudes: 10th percentile and median of '
      f'this dataset); RMSSD error evaluated at the 10th percentile and median reference RMSSD:')
print(floor.round(2).to_string(index=False))
print(f'rounding to the nearest sample alone: h/sqrt(12) = {1000 / OX_FS / np.sqrt(12):.2f} ms')
print()
gap = beat_audit.rmssd_peak_ms - beat_audit.rmssd_slope_ms
print(f'RMSSD from the systolic peak minus RMSSD from the steepest upstroke: '
      f'median {gap.median():+.1f} ms '
      f'(interquartile {gap.quantile(0.25):+.1f} to {gap.quantile(0.75):+.1f})')
print('recordings where the peak fiducial adds more than 10 ms:')
print(beat_audit.loc[gap > 10, ['subject', 'activity', 'rmssd_slope_ms', 'rmssd_peak_ms']]
      .round(1).to_string(index=False) if (gap > 10).any() else '  none')

timing floor on a simulated 60 Hz pulse train (beat amplitudes: 10th percentile and median of this dataset); RMSSD error evaluated at the 10th percentile and median reference RMSSD:
 amplitude  rounded_to_levels  sub_sample  timing_sd_ms  noise_rmssd_ms  error_at_18ms  error_at_41ms
        43              False       False          4.76           11.65           3.49           1.63
        43              False        True          0.16            0.39           0.00           0.00
        43               True       False          4.79           11.73           3.53           1.65
        43               True        True          0.49            1.20           0.04           0.02
        52              False       False          4.76           11.65           3.49           1.63
        52              False        True          0.16            0.39           0.00           0.00
        52               True       False          4.76           11.66           3.49           1.63
  

## 5. Heart-rate variability per activity
The reference's own HRV, from the kept intervals of the steepest-upstroke fiducial: mean heart
rate, SDNN, RMSSD and pNN50. Two cautions shape what is reported. Two minutes is an
ultra-short recording, enough for heart rate and RMSSD but not for the frequency-domain
measures, which want five. And breathing paced at 11 s per cycle puts its heart-rate swing at
0.09 Hz, inside the conventional low-frequency band, so an LF/HF ratio here would measure the
protocol rather than autonomic balance; none is computed.

This is pulse-rate variability at the finger, not R-R variability from an ECG. The two agree
closely at rest; deep breathing changes the pulse transit time with each breath, which shifts
finger beats against R waves by a few milliseconds. For a camera, which also sees the pulse
and not the R wave, the finger is the like-for-like reference.

Only references that can carry HRV are summarised, and two things rule one out. One is a
clock that the positive control in section 3 doubts: a loss missed or put at the wrong sample
dates a stretch one period off, which adds 16.7 ms steps to the successive differences that
RMSSD is built from. The other is a thin record, under 90% of its intervals kept or under
90 s of them: its gaps sit where the waveform was lost or clipped, and clipping can follow
the breathing or the hand's movement rather than fall at random.

Each activity is also compared with the same subject's stationary recording, where that one
carries HRV too, so differences between people do not masquerade as differences between
activities.

In [8]:
rows = []
for (subject, activity), table in BEATS.items():
    rows.append({'subject': subject, 'activity': activity, **hrv_metrics(table),
                 'kept_share': float(table.kept.mean()), 'kept_s': float(table.ibi[table.kept].sum())})
hrv_table = pd.DataFrame(rows)

# A reference carries HRV when the positive control in section 3 trusts its clock and it keeps at
# least 90% of its intervals and 90 s of them.
doubted = set(zip(shaky.subject, shaky.activity))
hrv_table['clock_doubted'] = [key in doubted for key in zip(hrv_table.subject, hrv_table.activity)]
hrv_table['thin'] = (hrv_table.kept_share < 0.9) | (hrv_table.kept_s < 90)
hrv_table['carries_hrv'] = ~hrv_table.clock_doubted & ~hrv_table.thin
usable = hrv_table[hrv_table.carries_hrv]
stationary = usable[usable.activity == 'stationary'].set_index('subject')
usable = usable.assign(rmssd_vs_stationary=usable.rmssd_ms / usable.subject.map(stationary.rmssd_ms))


def quartiles(s):
    return f'{s.median():.1f} ({s.quantile(0.25):.1f}-{s.quantile(0.75):.1f})'


print(f'references that carry HRV: {len(usable)} of {len(hrv_table)}')
print('set aside, clock doubted by the positive control:',
      [f'{r.subject}/{r.activity}' for r in hrv_table[hrv_table.clock_doubted].itertuples()] or 'none')
print('set aside, too thin (under 90% of intervals kept, or under 90 s of kept intervals):',
      hrv_table[hrv_table.thin][['subject', 'activity', 'kept_share', 'kept_s']].round(2)
      .to_dict('records') or 'none')
print()
summary = (usable.groupby('activity')
           .agg(recordings=('hr_bpm', 'size'),
                hr_bpm=('hr_bpm', quartiles), rmssd_ms=('rmssd_ms', quartiles),
                sdnn_ms=('sdnn_ms', quartiles), pnn50=('pnn50', 'median'),
                rmssd_vs_stationary=('rmssd_vs_stationary', 'median'), kept=('kept_share', 'median'))
           .reindex(list(ACTIVITIES)))
print('per activity over the references that carry HRV, median (interquartile range) over subjects;')
print('the ratio is to the same subject\'s stationary RMSSD, where that recording carries HRV too:')
print(summary.round(3).to_string())
print()
print(f'heart rate across all {len(hrv_table)} recordings '
      f'{hrv_table.hr_bpm.min():.0f}-{hrv_table.hr_bpm.max():.0f} bpm | RMSSD across the '
      f'{len(usable)} that carry HRV {usable.rmssd_ms.min():.0f}-{usable.rmssd_ms.max():.0f} ms')

references that carry HRV: 58 of 70
set aside, clock doubted by the positive control: ['005/reading', '005/stationary', '005/watching', '006/reading', '006/stationary', '013/watching', '015/watching']
set aside, too thin (under 90% of intervals kept, or under 90 s of kept intervals): [{'subject': '004', 'activity': 'deep', 'kept_share': 0.87, 'kept_s': 102.13}, {'subject': '004', 'activity': 'watching', 'kept_share': 0.88, 'kept_s': 105.09}, {'subject': '005', 'activity': 'talking', 'kept_share': 0.86, 'kept_s': 102.93}, {'subject': '006', 'activity': 'stationary', 'kept_share': 0.5, 'kept_s': 59.9}, {'subject': '008', 'activity': 'watching', 'kept_share': 0.86, 'kept_s': 95.59}, {'subject': '013', 'activity': 'watching', 'kept_share': 0.87, 'kept_s': 105.86}, {'subject': '014', 'activity': 'watching', 'kept_share': 0.88, 'kept_s': 103.17}]

per activity over the references that carry HRV, median (interquartile range) over subjects;
the ratio is to the same subject's stationary RMSSD, 

## 6. Deep breathing as the positive control
Paced at 6 s in and 5 s out, deep breathing should drive the heart rate up on every
inspiration and down on every expiration - respiratory sinus arrhythmia, at its largest at
slow breathing rates. If the reference cannot show that swing, it cannot support finer HRV
claims; if it shows it plainly, these recordings become the first test any camera method has
to pass.

Two measures, both from the kept intervals. The heart-rate series is put on a 4 Hz grid and
detrended, and its spectrum gives the dominant frequency between 0.04 and 0.4 Hz and the share
of power within 0.02 Hz of the paced rate, 1/11 Hz. The swing is the median, over consecutive
11 s windows, of the highest minus the lowest heart rate in the window. Both are computed for
every activity, so each subject's deep breathing is set against their own stationary
recording.

In [9]:
def rsa_measures(table, fs=HR_GRID_HZ, breath_s=BREATH_PERIOD_S):
    """Dominant frequency, paced-band share and per-breath swing of the heart-rate series."""
    kept = table.kept.to_numpy()
    t = table.t.to_numpy()[kept]
    hr = 60.0 / table.ibi.to_numpy()[kept]
    grid = np.arange(t[0], t[-1], 1.0 / fs)
    series = np.interp(grid, t, hr)
    detrended = series - np.polyval(np.polyfit(grid, series, 1), grid)
    power = np.abs(np.fft.rfft(detrended * np.hanning(grid.size))) ** 2
    freq = np.fft.rfftfreq(grid.size, 1.0 / fs)
    band = (freq >= 0.04) & (freq <= 0.4)
    paced = band & (np.abs(freq - 1.0 / breath_s) <= 0.02)
    width = int(round(breath_s * fs))
    swings = [series[i:i + width].max() - series[i:i + width].min()
              for i in range(0, grid.size - width + 1, width)]
    return {'peak_hz': float(freq[band][np.argmax(power[band])]),
            'paced_share': float(power[paced].sum() / power[band].sum()),
            'swing_bpm': float(np.median(swings))}


rsa = pd.DataFrame([{'subject': s, 'activity': a, **rsa_measures(table)}
                    for (s, a), table in BEATS.items()])
swing_by_subject = rsa.pivot(index='subject', columns='activity', values='swing_bpm')
share_by_subject = rsa.pivot(index='subject', columns='activity', values='paced_share')
deep = (rsa[rsa.activity == 'deep'].set_index('subject')
        .assign(breaths_per_min=lambda d: 60.0 * d.peak_hz,
                stationary_share=share_by_subject['stationary'],
                stationary_swing=swing_by_subject['stationary'],
                swing_ratio=lambda d: d.swing_bpm / d.stationary_swing))
on_pace = (deep.peak_hz - 1.0 / BREATH_PERIOD_S).abs() <= 0.02

print('deep breathing, per subject:')
print(deep[['peak_hz', 'breaths_per_min', 'paced_share', 'stationary_share', 'swing_bpm',
            'stationary_swing', 'swing_ratio']].round(3).to_string())
print()
print(f'dominant heart-rate rhythm within 0.02 Hz of the paced {1 / BREATH_PERIOD_S:.3f} Hz: '
      f'{int(on_pace.sum())}/{len(deep)} subjects; off pace: {list(deep.index[~on_pace]) or "none"}')
print(f'swing larger in deep breathing than in the same subject\'s stationary recording: '
      f'{int((deep.swing_ratio > 1).sum())}/{len(deep)} | '
      f'median ratio {deep.swing_ratio.median():.2f}, '
      f'median swing {deep.swing_bpm.median():.1f} against {deep.stationary_swing.median():.1f} bpm')
print(f'share of heart-rate power at the paced rate higher than in the same subject\'s '
      f'stationary recording: '
      f'{int((deep.paced_share > deep.stationary_share).sum())}/{len(deep)} | median '
      f'{deep.paced_share.median():.2f} against {deep.stationary_share.median():.2f}')
print()
print('all activities, medians over subjects:')
print(rsa.groupby('activity')[['peak_hz', 'paced_share', 'swing_bpm']].median()
      .reindex(list(ACTIVITIES)).round(3).to_string())

deep breathing, per subject:
         peak_hz  breaths_per_min  paced_share  stationary_share  swing_bpm  stationary_swing  swing_ratio
subject                                                                                                   
003        0.087            5.249        0.542             0.143     24.617            14.676        1.677
004        0.084            5.010        0.513             0.694     18.034             7.037        2.563
005        0.091            5.466        0.757             0.234     16.525             7.421        2.227
006        0.093            5.558        0.570             0.267     16.343             6.825        2.394
007        0.101            6.050        0.810             0.340     20.032             9.090        2.204
008        0.101            6.038        0.551             0.322     11.087             8.016        1.383
009        0.093            5.558        0.609             0.228     19.972             7.924        2.520
010     

## 7. Findings and decision

**Layout (confirmed).** 14 subjects x 5 activities: all 70 recordings present, 254,352 raw
frames (310.5 GiB), 3,387-4,019 per recording. Every frame is exactly one 8-bit 1280x1024 PGM
by size, the frame count equals all three per-frame logs in every recording, and the same 19
names sit in every folder.

**Frames are dated by the camera clock.** The camera's counter shows no dropped frame in any
recording, and its own clock runs at 30.017-30.018 fps with every interval between 33.27 and
33.36 ms. A uniform grid at each recording's own rate is never more than 0.046 ms off, so a
store can keep frames as a uniform sequence with one frame rate - the measured one, since
30.000 would drift 72 ms over two minutes. The PC's arrival stamps are prompt in most
recordings (0.09 ms above the envelope in the median one) but not in four: 005/reading arrives
74 ms late in the median and more than 10 ms late in 88% of its frames, 006/reading and
005/watching up to 46 and 26 ms late, and 015/reading stalls for up to 2.4 s in its last
seconds, after its oximeter link had failed. The camera clock, mapped onto the PC's through the
lower envelope, is what dates a frame. Exposure was set per recording, 16-30 ms, at gain 0.

**The oximeter record.** `px1_full.mat` holds the waveform (one byte per sample, top bit set,
0-127 once removed), the PC's read time of each sample and the sample count, and nothing else:
no pulse rate, SpO2 or beat marker. The text log equals `pulseOxTime` in all 70, and
`px1_partial.mat` is a leading part of the full record in all 70 (shorter in 12, by up to 265
samples), so only the full record is used.

**Its clock, rebuilt.** The device samples at 60 Hz (-15 ppm in the median recording), and the
reader lost 2,803 samples: 20 per recording in the median, one at a time in 62 recordings, at
most 14 in a row. Dating samples by their index alone would end the median recording 0.32 s
early, and the stamps wander 122 ms peak to peak around a straight line. The rebuilt clock fits
all 70, with no step down in any, and leaves 5.3 ms of read delay in the median recording. In
the positive control, 15,082 losses injected at three times the dataset's own rate came back
with the count off by 65 in all and 0.28 misplacements per run beyond the two-sample guard. The
rule mishandled more than 2% of them in seven recordings: six with slow or erratic reads
(median read delay 10-78 ms) - 005/stationary, 005/reading, 005/watching, 006/stationary,
006/reading and 015/watching - and 013/watching. The slowness was the PC's, not a device's:
005/reading's frames arrive 74 ms late and its oximeter samples 78 ms late.

**Coverage.** The reference starts 0.05-0.24 s after the first frame. In three recordings the
oximeter link failed before the camera stopped, leaving the last 8.4 s (003/reading), 7.4 s
(015/reading) and 3.2 s (009/stationary) of frames without a reference; those frames are left
out of any evaluation.

**Reference beats.** The steepest point of the upstroke is the fiducial. The systolic peak
agrees with it in most recordings (RMSSD +0.6 ms in the median) but not in seven, where the
maximum moves between the systolic and diastolic waves and RMSSD rises by 15 to 216 ms
(008/watching: 51 against 267 ms). With the minimum spacing tied to each recording's own beat
period, 96.7% of intervals are kept in the median recording; 499 are dropped next to a lost or
clipped sample, 24 for departing from their neighbours and 11 for range. A beat spans 52 of the
127 levels in the median recording, and the waveform sits at zero in 0.47% of samples, 8.2% at
worst (004/deep).

**The sampling is not the limit.** On a simulated train at this dataset's amplitudes, the
steepest upstroke refined between samples is timed to 0.35-0.49 ms after 7-bit rounding, which
adds 0.02-0.04 ms to an RMSSD of 18 ms. Without the refinement the same sampling costs 4.8 ms
per beat and 3.5 ms of RMSSD at 18 ms, a 20% error; taking the 11.7 ms noise RMSSD as the
error, as the sampling-ceiling script's `predicted` column does, would overstate it threefold.
Sub-sample refinement is essential and enough: whatever limits HRV from this reference, it is
not 60 Hz or 7 bits.

**HRV in the reference.** Pulse-rate variability at the finger over two minutes, from the 58
references that carry it: heart rate 50-111 bpm across all 70, RMSSD 8-96 ms. At rest the
median subject shows an RMSSD of 40 ms (interquartile 23-55) and an SDNN of 42 ms. Deep
breathing raises them to 55 and 78 ms, 1.24 times the same subject's resting RMSSD; reading
raises RMSSD by 17%, watching and talking by 4-5%, and talking lifts SDNN to 65 ms. That is
more variability than the 20-27 ms resting RMSSD of the MCD-rPPG subjects behind the earlier
HRV verdict, which makes this the more forgiving test. For scale, a camera's RMSSD stays within
10% of a 40 ms reference only if its beat times carry less than about 7 ms of independent
error.

**Deep breathing, the positive control, passes.** In 13 of 14 subjects the heart rate's
dominant rhythm sits within 0.02 Hz of the paced 0.091 Hz, and in 13 of 14 its swing per 11 s
window is larger than in the same subject's resting recording: 17.3 against 9.0 bpm in the
median, 1.88 times. Subject 011 is the exception for the rhythm (8.6 per minute, with little
power at the paced rate) and 014 for the swing (11.9 against 12.2 bpm). The rhythm alone does
not single out deep breathing - reading, watching and talking also peak near 0.09 Hz in the
median subject - but the share of power at the paced rate does, 0.58 against 0.16-0.35. The
measured rates, 4.3-6.1 per minute among the 13 on pace, also make the protocol's 5.45 per
minute a target rather than a ground truth for breathing rate; at two minutes one spectral bin
is 0.5 per minute.

**Decision.**
- The reference supports heart rate in all 70 recordings, trimmed where the oximeter stopped
  early, and HRV in 58 (section 5). Out for HRV are the seven recordings where the control
  mishandled more than 2% of losses, since their rebuilt clocks can misdate a stretch by a
  period - 006/stationary among them, with a fifth of its samples lost and 60 s of usable
  intervals - and five more that keep under 90% of their intervals: 004/deep, 004/watching,
  005/talking, 008/watching and 014/watching. That leaves 12, 12, 8, 13 and 13 recordings
  for stationary, reading, watching, talking and deep breathing, and subject 005 with deep
  breathing alone. All 14 deep-breathing recordings stay in for heart rate and the breathing
  control.
- Beat times on both sides come from the steepest upstroke, refined between samples; the
  systolic peak is not used.
- Frames are dated by the camera clock at each recording's measured rate, and the reference by
  its rebuilt clock, both on the PC's clock. The camera-to-finger offset is still estimated per
  recording, with a search wide enough for the oximeter's own delay as well as the pulse's
  travel to the finger, and it should come out much the same in every recording.
- The roles stay evaluation ones. Which of them the dataset can carry now depends on the
  frames: their Bayer layout and clipping, the face and its motion per activity, and whether a
  camera pulse can be timed against this reference at all. The frame sections settle that.